In [ ]:
"""
Caltech ACN-Data 전체 다운로드 스크립트
============================================
사용 전 준비:
  pip install acnportal pandas tqdm

토큰 발급: https://ev.caltech.edu/dataset 에서 회원가입 후 발급
"""

import os
import time
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from acnportal.acndata import DataClient
from dotenv import load_dotenv

load_dotenv()

# ──────────────────────────────────────────
# ① 설정 (여기만 수정하면 됩니다)
# ──────────────────────────────────────────
API_TOKEN  = os.getenv("ACN_API_TOKEN")  # .env 파일
SITES      = ["caltech", "jpl", "office001"] # 수집할 사이트 목록
START_DATE = datetime(2018, 1, 1)            # 수집 시작일 (ACN 가장 초기 데이터)
END_DATE   = datetime.today()                # 오늘까지 전부
TIMESERIES = False   # True 로 바꾸면 분당 전류 시계열도 포함 (매우 느림)
OUTPUT_DIR = "acn_data"                      # CSV 저장 폴더
# ──────────────────────────────────────────


def flatten_session(session: dict) -> dict:
    """
    중첩된 JSON 세션 딕셔너리를 단층(flat) 딕셔너리로 변환.
    timeseries 필드는 문자열로 직렬화해서 보존.
    """
    flat = {}
    for k, v in session.items():
        if isinstance(v, (list, dict)):
            flat[k] = str(v)   # 시계열 배열 등은 문자열로 저장
        else:
            flat[k] = v
    return flat


def download_site(client: DataClient, site: str) -> pd.DataFrame:
    """사이트 전체 세션을 다운로드해 DataFrame 반환."""

    # 총 세션 수 먼저 확인
    try:
        total = client.count_sessions(site)
        print(f"\n[{site}] 총 세션 수: {total:,}개")
    except Exception:
        total = None
        print(f"\n[{site}] 세션 수 조회 실패 — 그냥 진행합니다")

    records = []
    gen = client.get_sessions_by_time(
        site,
        start=START_DATE,
        end=END_DATE,
        timeseries=TIMESERIES,
    )

    pbar = tqdm(total=total, desc=f"  {site}", unit="세션")
    for session in gen:
        records.append(flatten_session(session))
        pbar.update(1)

        # API 서버 부하 완화: 1000세션마다 0.5초 대기
        if len(records) % 1000 == 0:
            time.sleep(0.5)

    pbar.close()
    return pd.DataFrame(records)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    client = DataClient(API_TOKEN)

    all_frames = []

    for site in SITES:
        try:
            df = download_site(client, site)
        except Exception as e:
            print(f"[{site}] 오류 발생: {e} — 건너뜁니다")
            continue

        if df.empty:
            print(f"[{site}] 데이터 없음")
            continue

        # ── 날짜 파싱 ──
        for col in ["connectionTime", "disconnectTime", "doneChargingTime"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

        # ── 사이트별 CSV 저장 ──
        out_path = os.path.join(OUTPUT_DIR, f"{site}.csv")
        df.to_csv(out_path, index=False, encoding="utf-8-sig")  # utf-8-sig = 엑셀 호환
        print(f"  → 저장 완료: {out_path}  ({len(df):,}행 × {len(df.columns)}열)")

        all_frames.append(df)

    # ── 전체 통합 CSV 저장 ──
    if all_frames:
        combined = pd.concat(all_frames, ignore_index=True)
        combined_path = os.path.join(OUTPUT_DIR, "acn_all_sites.csv")
        combined.to_csv(combined_path, index=False, encoding="utf-8-sig")
        print(f"\n✅ 통합 파일 저장: {combined_path}  ({len(combined):,}행)")

        # ── 간단 요약 출력 ──
        print("\n📊 데이터 요약:")
        print(f"  기간: {combined['connectionTime'].min()} ~ {combined['connectionTime'].max()}")
        print(f"  사이트별 세션 수:\n{combined['siteID'].value_counts().to_string()}")
        if "kWhDelivered" in combined.columns:
            print(f"  총 충전량: {combined['kWhDelivered'].sum():,.1f} kWh")


if __name__ == "__main__":
    main()